# PFAS Literature RAG - local workflow check

This notebook runs a compact end-to-end check of the local literature RAG workflow:

1. inspect the local corpus and document manifest;
2. confirm the index is available;
3. run hybrid vector/BM25 retrieval;
4. show citation-bearing context;
5. generate a short local answer with Ollama.

The notebook assumes PDFs have already been collected or copied into `data/raw_pdfs/` and indexed with `uv run pfas-ingest`.

In [1]:
# ruff: noqa: E402, I001
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

from pfas_lit_rag.answering import answer_question
from pfas_lit_rag.config import get_settings
from pfas_lit_rag.document_manifest import DocumentManifest
from pfas_lit_rag.retrieval import format_context, search_index
from pfas_lit_rag.vector_store import VectorStore

settings = get_settings()
settings

Settings(project_root=PosixPath('/home/laura/Documents/RAG Litterature Scientifique PFAS'), raw_pdf_dir=PosixPath('data/raw_pdfs'), index_dir=PosixPath('data/index'), metadata_dir=PosixPath('data/metadata'), embedding_model='BAAI/bge-small-en-v1.5', ollama_base_url='http://localhost:11434', ollama_model='qwen2.5:3b', chunk_size=900, chunk_overlap=150, retrieval_k=4, lexical_candidate_k=20, vector_weight=0.65, lexical_weight=0.35, context_chars_per_chunk=1200, ollama_num_predict=350, request_timeout_seconds=900.0, collector_user_agent='pfas-lit-rag/0.1')

## Local corpus state

In [2]:
pdfs = sorted(settings.resolved_raw_pdf_dir.glob("*.pdf"))
manifest = DocumentManifest(settings.resolved_metadata_dir).load()
store = VectorStore(settings.resolved_index_dir)
_, chunks = store.load()

print(f"PDF files: {len(pdfs)}")
print(f"Manifest records: {len(manifest)}")
print(f"Indexed chunks: {len(chunks)}")
print(f"Index has duplicate chunk fingerprints: {store.has_duplicate_fingerprints()}")

PDF files: 18
Manifest records: 18
Indexed chunks: 469
Index has duplicate chunk fingerprints: False


In [3]:
for record in list(manifest.values())[:5]:
    print(f"- {record.title}")
    print(f"  pages={record.pages_extracted}, chunks={record.chunks_indexed}")
    print(f"  sha256={record.file_sha256[:12]}...")

- Comparison of extraction methods for per- and polyfluoroalkyl substances (PFAS) in human serum and placenta samples—insights into extractable organic fluorine (EOF)
  pages=12, chunks=13
  sha256=48a0fa8b949a...
- EOF and target PFAS analysis in surface waters affected by sewage treatment effluents in Berlin, Germany
  pages=10, chunks=13
  sha256=2ec7af5d7121...
- Less is more: a methodological assessment of extraction techniques for per- and polyfluoroalkyl substances (PFAS) analysis in mammalian tissues
  pages=14, chunks=17
  sha256=ed9e659a2cab...
- Evaluation of extraction methodologies for PFAS analysis in mascara: a comparative study of SPME and automated µSPE
  pages=14, chunks=15
  sha256=8a7be56b17c7...
- Comparing PFAS analysis in batch leaching and column leaching tests
  pages=19, chunks=24
  sha256=5026cbb5d613...


## Hybrid retrieval

The search combines dense retrieval from FAISS with a lightweight BM25 lexical score. This helps exact technical terms such as `LC-MS/MS`, `SPE-WAX`, `PFOA`, or `GenX` remain visible.

In [4]:
question = "What analytical methods are used for PFAS detection?"
results = search_index(question, settings=settings, top_k=4)

for i, result in enumerate(results, start=1):
    print(f"[{i}] score={result.score:.3f} | {result.citation}")
    print(result.chunk.text[:450].replace("\n", " "))
    print()

[1] score=0.987 | Current progress in the environmental analysis of poly- and perfluoroalkyl substances (PFAS), p. 6
mass spectrometry (HRMS), novel PFASs have been continu- ously identi ed in a variety of environmental matrices. 15 Recently, time-of- ight mass spectrometry (TOF-MS), Fourier transform ion cyclotron resonance mass spectrometry (FTICR- MS) and orbitrap instrumentations have been utilized for structure proposal of unknown PFAS molecules from more than twenty established classes 16 and contribute to the developing eld of emerging PFAS discovery 

[2] score=0.566 | Evaluation of extraction methodologies for PFAS analysis in mascara: a comparative study of SPME and automated µSPE, p. 1
Vol.:(0123456789) Analytical and Bioanalytical Chemistry (2025) 418:619–632 https://doi.org/10.1007/s00216-025-05908-x RESEARCH PAPER Evaluation of extraction methodologies for PFAS analysis in mascara: a comparative study of SPME and automated µSPE Aghogho A. Olomukoro1 · Lucas Lüthy2 · To

## Context sent to Ollama

The generation step receives a compact context with numbered passages. The final answer should cite these passage numbers and list source pages.

In [5]:
print(format_context(results[:2], max_chars_per_chunk=900))

[1] Current progress in the environmental analysis of poly- and perfluoroalkyl substances (PFAS), p. 6
Score: 0.987
mass spectrometry (HRMS), novel PFASs have been continu- ously identi ed in a variety of environmental matrices. 15 Recently, time-of- ight mass spectrometry (TOF-MS), Fourier transform ion cyclotron resonance mass spectrometry (FTICR- MS) and orbitrap instrumentations have been utilized for structure proposal of unknown PFAS molecules from more than twenty established classes 16 and contribute to the developing eld of emerging PFAS discovery via suspect and non-target screening.17 The objective of the present review was to investigate the newest monitoring trends and the most recent analytical methods for the identi cation and determination of PFASs in diverse environmental matrices, including untargeted and non- specic approaches. The advantages and shortcomings of the existing analytical techniques and their performance with several sample types are discussed in d

## Local answer generation

This cell calls the local Ollama model configured in `PFAS_RAG_OLLAMA_MODEL` or `qwen2.5:3b` by default. On CPU-only hardware it may take a few minutes.

In [6]:
response = answer_question(question, settings=settings, top_k=2)
print(response.answer)

The retrieved passages discuss different analytical methods used for detecting poly- and perfluoroalkyl substances (PFAS). 

[1] mentions that mass spectrometry (HRMS), time-of-flight mass spectrometry (TOF-MS), Fourier transform ion cyclotron resonance mass spectrometry (FTICR-MS) and orbitrap instrumentation have been utilized for structure proposal of unknown PFAS molecules. These methods are used to identify and determine PFAS in diverse environmental matrices, including untargeted and non-specific approaches.

[2] discusses the evaluation of extraction methodologies for PFAS analysis in mascara products. It compares two specific methods: Solid-phase microextraction (SPME) and automated µSPE (micro-solid phase extraction). 

Therefore, based on these passages, analytical methods used for PFAS detection include mass spectrometry techniques such as HRMS, TOF-MS, FTICR-MS, orbitrap, SPME, and automated µSPE.

Sources:
[1] Current progress in the environmental analysis of poly- and per

In [7]:
print("Retrieved citations:")
for citation in response.citations:
    print(f"- {citation}")

Retrieved citations:
- Current progress in the environmental analysis of poly- and perfluoroalkyl substances (PFAS), p. 6
- Evaluation of extraction methodologies for PFAS analysis in mascara: a comparative study of SPME and automated µSPE, p. 1
